In [ ]:
import os
import sys
from pathlib import Path
import importlib
import torch

In [ ]:
#@title Setup
root_path = "/content/drive/MyDrive/MSc/Flood-Mapping"  #@param {type:"string", multiline:true}
Dataset_url = "https://huggingface.co/datasets/tax2310/STURM-fusion-24/resolve/main/Dataset.zip"  #@param {type:"string", multiline:true}
mount_drive = False  #@param {type:"boolean"}
clone_repo = True  #@param {type:"boolean"}
download_results = True  #@param {type:"boolean"}
run_training = False  #@param {type:"boolean"}

learning_rates = [1e-3, 1e-4]
batch_sizes = [32, 64]
weight_decays = [0.0, 1e-5]
dropout_rates = [0.0, 0.2]
num_workers = 8


import sys
from pathlib import Path

REPO_URL = "https://github.com/TAX2310/Flood-Mapping.git"

if not mount_drive and not clone_repo:
    raise ValueError("Either mount_drive or clone_repo must be True.")

if mount_drive:
    from google.colab import drive
    drive.mount("/content/drive")
else:
    root_path = "Flood-Mapping"

repo = Path(root_path)

if clone_repo and not repo.exists():
    !git clone $REPO_URL $root_path

assert repo.exists(), f"Repo not found at {repo}. Enable clone_repo or fix root_path."

sys.path.append(str(repo))
from src.config import Fusion_CFG

cfg = Fusion_CFG()
cfg.ROOT = repo
cfg.DATASET_URL = Dataset_url
cfg.DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

#cfg.EPOCHS = 1


In [ ]:
requirements = cfg.ROOT / "requirements.txt"
!pip install -r {requirements}

In [ ]:
import src.data.sturm_fusion as SturmFusion
import src.train.training as training
import src.test.testing as testing
import src.util.io as io
import src.util.plotting as plot

In [ ]:
!pip install wget

In [ ]:

import zipfile
import wget

def bar_progress(current, total, width=80):
    progress = current / total * 100
    print(f"\rDownloading: {progress:.1f}% [{current}/{total} bytes]", end="")

def download_and_extract_dataset(cfg):
    url = cfg.DATASET_ZIP_URL
    data_path = cfg.DATA_PATH
    dataset_zip_path = cfg.DATASET_ZIP_PATH
    sentinel1_dir = cfg.S1_PATH
    mask_dir = cfg.MASK_PATH

    os.makedirs(data_path, exist_ok=True)
    is_extracted = sentinel1_dir.exists() and mask_dir.exists()

    # 2. Download if not already present
    if not dataset_zip_path.exists() and not is_extracted:
        print("Downloading dataset...")
        wget.download(url, bar=bar_progress)
    else:
        print("Zip or Dataset already exists, skipping download.")

    # 3. Check if already extracted

    if is_extracted:
        print("Dataset already extracted, skipping unzip.")
    else:
        print("Extracting dataset...")

        extract_path = cfg.ROOT.resolve()  # forces correct absolute path

        print(f"Extracting to: {extract_path}")

        with zipfile.ZipFile(cfg.DATASET_ZIP_PATH, 'r') as zip_ref:
            zip_ref.extractall(extract_path)

        print("Extraction complete.")

    # 4. Delete zip to save space
    if dataset_zip_path.exists():
        dataset_zip_path.unlink()
        print("Zip file deleted.")

    # 5. Final check
    print("\nFinal structure:")
    for p in data_path.iterdir():
        print(" -", p.name)

    return data_path

In [ ]:
#data_root = SturmFusion.download_and_extract_dataset(cfg)

download_and_extract_dataset(cfg)

img_dir = cfg.S1_PATH
mask_dir = cfg.MASK_PATH

print("Image dir exists:", img_dir.exists())
print("Mask dir exists:", mask_dir.exists())

print("Num images:", len(list(img_dir.glob("*.tif"))))
print("Num masks:", len(list(mask_dir.glob("*.tif"))))

if download_results:
    SturmFusion.download_and_extract_results(cfg)

In [ ]:
if run_training:
    for learning_rate in learning_rates:
        for batch_size in batch_sizes:
            for weight_decay in weight_decays:
                for dropout_rate in dropout_rates:
                    cfg.LR = learning_rate
                    cfg.BATCH_SIZE = batch_size
                    cfg.WEIGHT_DECAY = weight_decay
                    cfg.DROPOUT_RATE = dropout_rate
                    training.train_from_file(cfg, num_workers=num_workers)

In [ ]:
plot.plot_hp_comparison_bar(cfg, save_path=cfg.FIG_EXPORTS_DIR/"fusion_hp_iou_f1.pdf")

In [ ]:
plot.view_training_metrics(cfg)

In [ ]:
testing.select_model_to_test(cfg)

In [ ]:
import src.inference.inference as inference

#samples = ["EMSR470_AOI01_46_07_2_1.tif", "EMSR441_AOI05_2_3_2_2.tif", "EMSR570_AOI02_07_03_2_1.tif"]

samples = ["EMSR470_AOI01_29_13_2_2.tif", "EMSR407_AOI01_03_17_2_1.tif", "EMSR470_AOI01_10_25_1_1.tif", "EMSR470_AOI01_47_10_1_2.tif", "EMSR629_AOI01_09_01_1_2.tif"]

results = inference.inference(cfg, cfg.FUSION_MODEL, samples)
#all_results = inference.inference(cfg, cfg.FUSION_MODEL)
#io.create_inference_results_csv(cfg, all_results, cfg.METADATA_CSV, cfg.FUSION_TEST_RESULTS_CSV)

In [ ]:
plot.plot_sample_results(cfg, results)

In [ ]:
plot.plot_metric_distribution_from_csv(cfg.FUSION_TEST_RESULTS_CSV, save_path=cfg.FIG_EXPORTS_DIR/"fusion_iou_dist.pdf")

In [ ]:
plot.plot_iou_vs_flood_scatter([cfg.FUSION_TEST_RESULTS_CSV], save_path=cfg.FIG_EXPORTS_DIR/"fusion_scatter.pdf", show_legend=False)

In [ ]:
plot.plot_average_iou_per_event(cfg.FUSION_TEST_RESULTS_CSV, save_path=cfg.FIG_EXPORTS_DIR/"fusion_iou_per_event.pdf")

In [ ]:
plot.plot_iou_vs_flood_scatter([cfg.S2_TEST_RESULTS_CSV,
                                cfg.S1_TEST_RESULTS_CSV,
                                cfg.FUSION_TEST_RESULTS_CSV], figsize=(10, 10), save_path=cfg.FIG_EXPORTS_DIR/"scatter_iou_flood_all.pdf", show_legend=True)

In [ ]:
plot.plot_fusion_improvement_distribution([cfg.S2_TEST_RESULTS_CSV,
                                cfg.S1_TEST_RESULTS_CSV,
                                cfg.FUSION_TEST_RESULTS_CSV], save_path=cfg.FIG_EXPORTS_DIR/"fusion_improvement_distribution.pdf")